# Running Forecast command to assess if it works correctly

In [1]:
import subprocess
import sys

In [ ]:


forecast_data = [
    [
        sys.executable,
        "-m",
        "MASTER.MASTER_forecast_fields",
        "--init-date",
        "2026-04-01",
        "--lead-months",
        "6",
        "--generate-config",
        "--active-months",
        "6",
        "7",
        "8",
        "9",
        "10",
        "11",
    ]
]

forecast_sample = [
    [
        sys.executable,
        "-m",
        "MASTER.MASTER_storm_parallel",
        "--forecast",
        f"forecast_configs/config_m0.json",
        # f"forecast_configs/config_m{m}.json",
        "--basins",
        "NA",
        "--years",
        "100",
        "--loop",
        "1",
        "--workers",
        "1",
    ]
    for m in range(48)
]


commands = (
    [
        # [sys.executable, "-m", "MASTER.MASTER_climatology"],
        # [sys.executable, "-m", "MASTER.MASTER_preprocessing"],
    ]
    + forecast_data
    + forecast_sample
)


In [ ]:
for cmd in forecast_sample:
    print("Running:", " ".join(cmd))
    subprocess.run(cmd, check=True)


# Visualizing Forecasts

## Get forecast outputs

In [ ]:
from POST_PROCESSING.utils.climada_utils import from_simulations_storm_mod

forecast_m = [
    from_simulations_storm_mod(
        f"/home/mbaldacchino/data/TSTFCST/STORM_DATA_IBTRACS_NA_FORECAST_m{m}_100_YEARS_0.txt"
    )
    for m in range(17)
]


## Compute counts

In [ ]:
# Build summary year
import pandas as pd
from math import inf

SS_CAT = pd.DataFrame(
    {
        "Cat": ["TS", "C1", "C2", "C3", "C4", "C5"],
        "Wind_min": [35, 64, 83, 96, 113, 137],
        "Wind_ax": [64, 83, 96, 113, 137, inf],
    }
)
COUNTS = []
for j in range(len(forecast_m)):  # iterate over members SEAS5
    forecast_j = forecast_m[j].data
    for year in range(100):  # iterate over each year
        forecast_j_year = [
            forecast_j[i].max_sustained_wind.max().to_numpy()
            for i in range(len(forecast_j))
            if int(forecast_j[i].name.split("-")[1]) == year
        ]
        if len(forecast_j_year) > 0:
            COUNTS.append(
                [
                    len(
                        [
                            forecast_j_year[k]
                            for k in range(len(forecast_j_year))
                            if forecast_j_year[k] >= SS_CAT.iloc[i, 1]
                            # and forecast_j_year[k] < SS_CAT.iloc[i, 2]
                        ]
                    )
                    for i in range(SS_CAT.shape[0])
                ]
            )
        else:
            COUNTS.append([0] * 6)

counts_df = pd.DataFrame(COUNTS, columns=SS_CAT["Cat"])

## Plot counts

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import gaussian_kde
from matplotlib.lines import Line2D
from matplotlib.ticker import MultipleLocator


def plot_storm_count_distributions(
    counts_df: pd.DataFrame,
    columns=("TS", "C1", "C2", "C3", "C4", "C5"),
    stats=("median", "mode"),
    bw_adjust=2.2,  # larger = smoother KDE
    fill=True,
    sharex=True,
):
    """
    Plot one KDE per storm category with vertical bars from y=0 up to the KDE
    at the requested summary statistics.

    Parameters
    ----------
    counts_df : pd.DataFrame
        Rows = years, columns = storm categories.
    columns : tuple/list
        Column order to plot.
    stats : tuple/list
        Supported: 'median', 'mode', 'mean'
    bw_adjust : float
        Multiplier on Scott bandwidth. Increase to smooth more for discrete counts.
    fill : bool
        Fill under each KDE.
    sharex : bool
        Share x-axis across subplots.
    """

    # Approximate Saffir-Simpson-like palette from TS to Cat 5
    colors = {
        # "TD": "#7FFFD4",
        "TS": "#18DCE0",  # tropical storm cyan
        "C1": "#E8E7B8",  # pale yellow
        "C2": "#F0DE73",  # warm yellow
        "C3": "#F4BE45",  # golden
        "C4": "#FF8F1F",  # orange
        "C5": "#FF6165",  # coral red
    }

    stat_styles = {
        "median": {"linestyle": "--", "linewidth": 2.2},
        "mode": {"linestyle": "-", "linewidth": 2.4},
        "mean": {"linestyle": ":", "linewidth": 2.2},
    }

    plot_cols = [c for c in columns if c in counts_df.columns]
    if not plot_cols:
        raise ValueError("None of the requested columns were found in counts_df.")

    n = len(plot_cols)
    fig, axes = plt.subplots(n // 2, 2, figsize=(14, n), sharex=sharex, squeeze=False)
    axes = axes.ravel()

    global_xmin = np.inf
    global_xmax = -np.inf
    kde_info = []

    # First pass: compute KDEs and x-ranges
    for col in plot_cols:
        x = counts_df[col].dropna().astype(float).to_numpy()
        if len(x) == 0:
            kde_info.append((col, None, None, None))
            continue

        xmin, xmax = x.min(), x.max()
        global_xmin = min(global_xmin, xmin)
        global_xmax = max(global_xmax, xmax)

        kde_info.append((col, x, xmin, xmax))

    if not np.isfinite(global_xmin):
        raise ValueError("All selected columns are empty.")

    xpad = max(
        1.0, 0.12 * (global_xmax - global_xmin if global_xmax > global_xmin else 5)
    )
    xs = np.linspace(max(0, global_xmin - xpad), global_xmax + xpad, 500)
    # Legend handles for summary stats
    legend_handles = []
    for s in stats:
        if s in stat_styles:
            legend_handles.append(
                Line2D(
                    [0],
                    [0],
                    color="black",
                    linestyle=stat_styles[s]["linestyle"],
                    linewidth=stat_styles[s]["linewidth"],
                    label=s.capitalize(),
                )
            )

    for ax, (col, x, xmin, xmax) in zip(axes, kde_info):
        if x is None:
            ax.set_visible(False)
            continue

        color = colors.get(col, None)

        # Constant series: no KDE possible
        if np.allclose(x, x[0]):
            x0 = x[0]
            ax.vlines(x0, 0, 1.0, color=color, linewidth=2.5)
            ax.set_ylim(0, 1.05)
            ax.set_title(col, fontsize=12, loc="left", fontweight="bold")
            ax.set_ylabel("Density")
            ax.grid(alpha=0.2)
            continue

        # Stronger smoothing to reduce artificial multimodality for discrete counts
        kde = gaussian_kde(x, bw_method=lambda s: s.scotts_factor() * bw_adjust)
        ys = kde(xs)

        ax.plot(xs, ys, color=color, linewidth=2.6)
        if fill:
            ax.fill_between(xs, 0, ys, color=color, alpha=0.35)

        # Summary statistic locations
        stat_x = {}
        if "median" in stats:
            stat_x["median"] = np.median(x)
        if "mean" in stats:
            stat_x["mean"] = x.mean()
        if "mode" in stats:
            # mode of the smoothed KDE
            stat_x["mode"] = xs[np.argmax(ys)]

        # Draw vertical bars up to density height
        for stat_name, x0 in stat_x.items():
            y0 = float(kde([x0])[0])
            ax.vlines(x0, 0, y0, color="black", alpha=0.95, **stat_styles[stat_name])

        ax.set_title(col, fontsize=12, loc="left", fontweight="bold")
        ax.set_ylabel("Density")
        ax.grid(alpha=0.2)
        ax.set_xlabel("Annual count")
        ax.tick_params(axis="x", labelbottom=True)
        ax.xaxis.set_major_locator(MultipleLocator(1))
        ax.set_xlim((0, x.max()))

    if legend_handles:
        axes[0].legend(
            handles=legend_handles, loc="upper right", frameon=True, title="Statistic"
        )

    fig.suptitle(
        "Distribution of yearly storm counts by category", y=0.995, fontsize=14
    )
    fig.tight_layout()
    return fig, axes


fig, axes = plot_storm_count_distributions(
    counts_df,
    columns=["TS", "C1", "C2", "C3", "C4", "C5"],
    stats=("mean", "mode"),
    bw_adjust=2.2,
    fill=True,
    sharex=False,
)
plt.show()
